# Chapter 15 — Supplement 5: `draft_response`

*A LoRA adapter writes the reply; a GMS verifier checks the numbers and the promises.*

Companion to `15_capstone.ipynb`. The draft is the one field the agent *generates* rather than transcribes, and the commitment a customer actually sees — so it is the highest-value thing to verify. This notebook calls the shipped LoRA writer and the GMS draft verifier.

## Where it fits

Step 5, the final workflow step. For a `complaint` it runs the LoRA writer and then the GMS verifier; for `inquiry`/`other` it uses a fixed template, keeping the LoRA on its SFT distribution.

| | |
| --- | --- |
| **Backed by** | Qwen2.5-3B-Instruct + a PEFT LoRA (complaints); template (inquiry/other) |
| **Artifact** | `data/draft_response_lm_qwen/` (PEFT adapter) |
| **Training script** | `scripts/train_draft_response_lora.py` |
| **Verifier** | `glassloop/capstone/draft_verifier.py` (ENM + promise check) |
| **Risk level** | `MEDIUM` (it produces external-facing text) |

Two failure modes, two checks (chapter §"Verifying the draft"): a LoRA can **drift a digit** (fixed deterministically from Exact Numerical Memory), and a helpful model can **promise more than policy allows** (escalated to a human).

In [ ]:
import os, json, warnings, contextlib, io
from pathlib import Path

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('KNOWLYTIX_EULA_ACCEPTED', '1')  # silence the EULA reminder line
warnings.filterwarnings('ignore')  # quiet the harmless GPU-capability / HF notices

# knowlytix prints a one-line license banner (it names the licensee) to stderr
# on first import. The package has no flag to disable it, so import it once here
# with stderr/stdout captured; every later import is a cache hit and stays quiet,
# so the banner never lands in a baked cell output.
with contextlib.redirect_stderr(io.StringIO()), contextlib.redirect_stdout(io.StringIO()):
    try:
        import knowlytix.core  # noqa: F401
    except Exception:
        pass

root = Path('.')
if not (root / 'data').exists():
    root = Path('..')
assert (root / 'data').exists(), 'run this notebook from the repo root or notebooks/'
print('repo root:', root.resolve())

## 1. The LoRA writer

`get_default_lm()` loads the base Qwen with the PEFT LoRA adapter on top. `.generate(category, issue, policy_evidence, message)` returns a short, greedy-decoded reply. The adapter was SFT-trained to *write the reply* citing the policy — not to echo the prompt.

In [ ]:
from glassloop.models.draft_response_lm import get_default_lm

lm = get_default_lm()
draft = lm.generate(
    category='complaint',
    issue='overdraft_fee',
    policy_evidence=[{'id': 'overdraft'}],
    message='I was hit with a $35 overdraft fee and I want it reversed.',
)
print('draft:', repr(draft))

The reply is one or two sentences, on-template, and names the governing policy. Greedy decoding makes it deterministic for a fixed input — the one *generated* component of the agent's output, which is why it is verified before it leaves the agent.

## 2. The GMS draft verifier — correct the numbers (ENM)

The fee a draft cites has one authoritative value, held in the GMS store's **Exact Numerical Memory** (ENM) — a byte-exact register the model cannot paraphrase. The verifier parses the dollar amount in the draft, looks up the authoritative figure, and substitutes it if it drifted (only when the draft names a *single* amount, so it never mis-edits a reply that legitimately cites two figures).

In [ ]:
from glassloop.capstone.draft_verifier import get_default_verifier, _POLICY_FEE_ENM

verifier = get_default_verifier()
print('policy -> ENM key map:', _POLICY_FEE_ENM)
authoritative = verifier.store.lookup_enm('fee_schedule', 'overdraft/per_occurrence')
print('authoritative overdraft fee (ENM):', authoritative)

drifted = 'Per our overdraft policy, the fee is $30 per occurrence.'
result = verifier.verify(drifted, policy_id='overdraft')
print('\ndraft in  :', drifted)
print('draft out :', result['text'])
print('corrected :', result['corrected'])
print('details   :', result['corrections'])

The verifier reads the authoritative $35 from ENM, sees the draft drifted to $30, and rewrites it — recording the correction (`draft_value` vs `authoritative`) so the audit log shows both what the model wrote and what the store corrected it to. This is the byte-exact guarantee a generative model cannot make on its own.

## 3. The GMS draft verifier — escalate the promise

A drifted digit is a typo; an unauthorized promise to waive a fee is a policy violation, and the safe disposition for a violation is a human. The verifier reuses the semantic classifier (the same one in the Policy gate): if the draft reads as an unconditional fee-waiver promise, it escalates. Its modal sensitivity is what makes this safe — a conditional, policy-grounded draft is *not* a promise and passes.

In [ ]:
promise   = 'Good news — we will waive your overdraft fee in full as a courtesy.'
grounded  = 'Per the overdraft policy, the $35 fee may be reversed once per year.'

for label, text in [('unconditional promise', promise), ('conditional/grounded', grounded)]:
    v = verifier.verify(text, policy_id='overdraft')
    print(f'{label:22s} -> escalate={v["escalate"]!s:5s}  reason={v["reason"]!r}')

The unconditional "we will waive" escalates (`requires_escalation`); the conditional "may be reversed once per year" passes. The classifier reads the modal distinction a keyword list cannot — *will waive* is a commitment, *may be reversed per policy* is not.

## 4. The governed tool and the template branch

`draft_response.fn` routes by category: a `complaint` goes through the LoRA writer and the verifier (returning `requires_escalation` when the verifier objects); `inquiry`/`other` take a deterministic template. The verifier's two outcomes — silent correction, or escalation — both surface in the tool output.

In [ ]:
from glassloop.capstone.banking_tools import draft_response

# complaint -> LoRA + verifier
c = draft_response.input_schema(category='complaint', issue='overdraft_fee',
    policy_evidence=[{'id': 'overdraft'}],
    message='Reverse my $35 overdraft fee now.')
print('complaint ->', draft_response.fn(**c.model_dump()))

# inquiry -> template (LoRA not used; stays on its SFT distribution)
i = draft_response.input_schema(category='inquiry', issue='branch_hours',
    policy_evidence=[],
    message='What are your Saturday hours?')
print('\ninquiry   ->', draft_response.fn(**i.model_dump()))

## 5. How the adapter was trained

`data/draft_response_lm_qwen/` was produced by `scripts/train_draft_response_lora.py`:

- A PEFT **LoRA** (rank 16, alpha 32) on the attention projections (`q/k/v/o_proj`) of Qwen2.5-3B-Instruct — ~7.4M trainable parameters (0.24% of the model).
- SFT on the **212 complaint-shaped pairs of the MeMo v3 corpus** (`data/training/bank_policy/draft_response_memo_v3.jsonl`), with the **loss masked to the completion** so the adapter learns to write the reply, not echo the prompt.
- Every training pair is **grounded by construction**: built from the policy graph and validated against the policy goldens (cite the keyword, contain the exact figure, no forbidden unauthorized-commitment phrase). The generator is `benchmarks/draft_adapter/build_memo_data.py`.
- Trained gently — **5 epochs at lr 1e-4** — because an earlier 12-epoch run on 121 mostly FAQ-shaped pairs over-fit the adapter *below* the base model's own prompted quality. The fix was better data, not more training (see Appendix D, §"Training the draft adapter").
- Greedy decoding at inference makes drafts deterministic.

```bash
python -m benchmarks.draft_adapter.build_memo_data   # regenerate the corpus
python scripts/train_draft_response_lora.py --epochs 5 --rank 16 --alpha 32
```

On 12 held-out complaints this fine-tune is **grounded on 100%** of drafts vs. 42% for the same Qwen prompted and 75% for a prompted frontier model — the value of fine-tuning is groundedness, not prose (Appendix D, §"Two judges for the draft").

We can confirm the shipped artifact is a LoRA adapter, not a full fine-tune:

In [ ]:
adapter_dir = root / 'data' / 'draft_response_lm_qwen'
print('artifact files:', sorted(p.name for p in adapter_dir.iterdir()))
cfg_path = adapter_dir / 'adapter_config.json'
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text())
    for key in ('r', 'lora_alpha', 'target_modules', 'base_model_name_or_path'):
        print(f'  {key:26s}: {cfg.get(key)}')

The presence of `adapter_config.json` / `adapter_model.*` (rather than a full set of model weights) confirms this is a small PEFT adapter loaded on top of the shared base model — the same Qwen the other tools reuse.

## Summary

- `draft_response` = **a LoRA writer for complaints + a fixed template for everything else**, so the adapter stays on its training distribution.
- The GMS draft verifier closes the generative gap two ways: **ENM** corrects a drifted dollar amount byte-exactly, and the **semantic classifier** escalates an unauthorized fee-waiver promise.
- The adapter (`scripts/train_draft_response_lora.py`) trains ~7.4M parameters with completion-masked loss.

Next: **Supplement 6 — the harness and the gate stack**, where the five tools are registered and wrapped in three governance gates.